In [ ]:
# ============================================================
# AI-BASED STUDENT PRESENTATION PERFORMANCE ANALYZER
# GOOGLE COLAB - ONE CELL
# ============================================================

!pip -q install gradio==5.44.1 faster-whisper

import re
import gradio as gr
from faster_whisper import WhisperModel

print("Loading AI Speech Model...")

model = WhisperModel(
    "tiny",
    device="cpu",
    compute_type="int8"
)

print("✅ Model loaded!")


FILLER_WORDS = [
    "um", "uh", "erm", "hmm",
    "like", "actually", "basically",
    "you know", "so"
]


def analyze_presentation(audio):

    if audio is None:
        return "⚠️ Please upload or record an audio file."

    try:

        # Speech to Text
        segments, info = model.transcribe(
            audio,
            beam_size=1
        )

        segments = list(segments)

        transcript = " ".join(
            s.text.strip()
            for s in segments
        ).strip()

        if not transcript:
            return "❌ Could not detect speech."

        # Duration
        duration = info.duration

        # Words
        words = re.findall(
            r"\b[\w']+\b",
            transcript.lower()
        )

        word_count = len(words)

        # Speaking speed
        minutes = max(duration / 60, 0.01)

        wpm = int(
            word_count / minutes
        )

        # Filler words
        filler_count = 0
        filler_found = []

        lower_text = transcript.lower()

        for filler in FILLER_WORDS:

            count = len(
                re.findall(
                    r"\b" + re.escape(filler) + r"\b",
                    lower_text
                )
            )

            if count > 0:

                filler_count += count

                filler_found.append(
                    f"{filler}: {count}"
                )

        # Vocabulary
        unique_words = len(
            set(words)
        )

        vocabulary_score = min(
            100,
            int(
                unique_words /
                max(word_count, 1) * 150
            )
        )

        # Speed score
        if 110 <= wpm <= 160:
            speed_score = 100
        elif 90 <= wpm <= 180:
            speed_score = 85
        elif 70 <= wpm <= 200:
            speed_score = 70
        else:
            speed_score = 50

        # Filler score
        filler_ratio = (
            filler_count /
            max(word_count, 1)
        )

        filler_score = max(
            0,
            int(
                100 -
                filler_ratio * 500
            )
        )

        # Overall
        overall = int(
            (
                speed_score +
                filler_score +
                vocabulary_score
            ) / 3
        )

        # Performance level
        if overall >= 85:
            level = "Excellent 🌟"
        elif overall >= 70:
            level = "Good 👍"
        elif overall >= 50:
            level = "Average 🙂"
        else:
            level = "Needs Improvement 📈"

        fillers = (
            ", ".join(filler_found)
            if filler_found
            else "No major filler words detected"
        )

        result = f"""
# 🎤 Presentation Performance Report

## 🏆 Overall Score

# {overall}/100

### Performance Level
**{level}**

---

## 📝 Transcript

{transcript}

---

## 📊 Speech Analysis

| Metric | Result |
|---|---|
| Speech Duration | {duration:.1f} seconds |
| Total Words | {word_count} |
| Speaking Speed | {wpm} WPM |
| Unique Words | {unique_words} |
| Filler Words | {filler_count} |
| Vocabulary Score | {vocabulary_score}/100 |
| Speed Score | {speed_score}/100 |
| Fluency Score | {filler_score}/100 |

---

## 🗣️ Filler Word Analysis

{fillers}

---

## 💡 Suggestions

"""

        if wpm < 90:
            result += "- Speak slightly faster to maintain audience attention.\\n"

        elif wpm > 180:
            result += "- Slow down your speech for better clarity.\\n"

        else:
            result += "- Your speaking speed is in a good range.\\n"

        if filler_count > 5:
            result += "- Reduce filler words such as 'um', 'uh', and 'like'.\\n"
        else:
            result += "- Good control over filler words.\\n"

        if vocabulary_score < 60:
            result += "- Try using a wider range of vocabulary.\\n"
        else:
            result += "- Your vocabulary diversity is good.\\n"

        return result

    except Exception as e:

        return f"❌ Error: {e}"


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Presentation Performance Analyzer"
) as app:

    gr.Markdown("""
# 🎤 AI-Based Student Presentation Performance Analyzer

### Analyze your presentation using Speech & Text Analysis.

**Upload/record → Speech-to-Text → NLP Analysis → Performance Score**
""")

    audio = gr.Audio(
        sources=["upload", "microphone"],
        type="filepath",
        label="🎙️ Upload or Record Presentation"
    )

    button = gr.Button(
        "🔍 Analyze Presentation",
        variant="primary"
    )

    output = gr.Markdown()

    button.click(
        analyze_presentation,
        inputs=audio,
        outputs=output
    )

    gr.Markdown("""
---
### TSA Concepts
Speech-to-Text • Text Analysis • Fluency Analysis •
Vocabulary Analysis • Filler Word Detection • Performance Scoring
""")


print("🚀 Launching public application...")

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/75.5M [00:00<?, ?B/s]

✅ Model loaded!
🚀 Launching public application...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://70b0fd593076188040.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
